# Experiment: Case-001 Record Request Matrix

Objective: prioritize public-safe record requests using labels only. Raw records, identifiers, exact dates, and treatment decisions stay out of this notebook.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass


@dataclass(frozen=True)
class RequestGroup:
    """Public-safe private-record request group."""

    name: str
    current_label: str
    evidence_weight: int
    privacy_risk: int
    uses_template: bool
    blocks_referral: bool

    def priority_score(self) -> int:
        """Rank groups that unblock clinician review while penalizing privacy risk."""
        score = self.evidence_weight

        if self.blocks_referral:
            score += 3

        if self.uses_template:
            score += 1

        return score - self.privacy_risk

In [ ]:
request_groups = [
    RequestGroup("diagnosis_and_genotype", "phenotype_only", 6, 2, False, True),
    RequestGroup("transfusion_burden", "transfusion_burden_unknown", 5, 2, True, True),
    RequestGroup(
        "immune_blood_bank", "immune_transfusion_packet_missing", 4, 2, True, True
    ),
    RequestGroup("iron_chelation_organ_risk", "iron_packet_missing", 4, 2, True, True),
    RequestGroup(
        "advanced_therapy_referral_readiness",
        "advanced_therapy_referral_packet_missing",
        2,
        2,
        True,
        True,
    ),
    RequestGroup(
        "private_source_index", "private_intake_index_needed", 3, 3, True, True
    ),
    RequestGroup("public_release_review", "release_check_required", 2, 1, True, False),
]

ranked_groups = sorted(
    request_groups,
    key=lambda group: (-group.priority_score(), group.name),
)

decision = {
    "matrix_label": "case001_record_request_matrix_active",
    "top_request_groups": [group.name for group in ranked_groups[:4]],
    "public_case_update_allowed": False,
    "private_storage_required": True,
    "patient_relevance_status": "patient_relevance_blocked",
}

decision

## Result

The top request groups are diagnosis/genotype, transfusion burden, immune/blood-bank, and iron/chelation/organ-risk context. This is workflow routing, not medical interpretation.
